In [ ]:
from planner import solve_foundation

result = solve_foundation(timeout=10.0, focus_area="Artificial Intelligence")

The following code is to fetch data from the nusmods api. 

Warning: It fetches ALL the courses data and takes around 20-30 mins to finish executing. Do not run unless sure you want it all. The `nusmods_modules.json` file already has the necessary courses for the first version of the project

In [ ]:
import json
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

BASE_URL = "https://api.nusmods.com/v2"
ACAD_YEAR = "2026-2027"
OUTPUT_FILE = Path("nusmods_modules.json")
TIMEOUT = 30


def get_json(url):
    response = requests.get(url, timeout=TIMEOUT)
    response.raise_for_status()
    return response.json()


def fetch_module(code):
    url = f"{BASE_URL}/{ACAD_YEAR}/modules/{code}.json"

    try:
        details = get_json(url)

        semester_data = details.get("semesterData", [])

        # Keep one entry per module, with its semesters in a list.
        return {
            "module_code": details.get("moduleCode", code),
            "module_credit": details.get("moduleCredit"),
            "semesters": [
                {
                    "semester_number": sem.get("semester"),
                    "exam_date_time": sem.get("examDate"),
                    "exam_duration": sem.get("examDuration"),
                }
                for sem in semester_data
            ],
            "preclusions": details.get("preclusion"),
            "prereq_tree": details.get("prereqTree"),
        }

    except requests.RequestException as error:
        print(f"Failed to fetch {code}: {error}")
        return None


print(json.dumps(fetch_module("CS3242"), indent=4))
